# SIPTA Notebook: Validación de datos — Educación

## Dominio: Educación

Este notebook valida la calidad estructural, lógica y territorial del dataset
**Oferta de cupos del sector oficial en Bogotá D.C.**

### Objetivos

- Verificar dimensiones y esquema del dataset.
- Identificar valores nulos.
- Detectar duplicados exactos.
- Validar la coherencia de las variables de oferta de cupos.
- Examinar la cardinalidad de establecimientos e identificadores DANE.
- Validar los códigos de las 20 localidades.
- Evaluar la consistencia entre `COD_LOCA` y la geometría publicada.
- Documentar excepciones territoriales sin modificar el archivo original.

> Esta etapa no realiza limpieza, corrección de registros, integración territorial ni cálculo de indicadores.

In [23]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent

EDUCACION_DIR = PROJECT_ROOT / "data" / "raw" / "EDUCACION"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"

archivo_educacion = EDUCACION_DIR / "ofertacupos_032025.geojson"
archivo_localidades = EXTERNAL_DIR / "loca.json"

gdf_educacion = gpd.read_file(archivo_educacion)
gdf_localidades = gpd.read_file(archivo_localidades)

print("=== CARGA PARA VALIDACIÓN ===")
print("Educación:", gdf_educacion.shape)
print("Localidades:", gdf_localidades.shape)

print("\nCRS Educación:", gdf_educacion.crs)
print("CRS Localidades:", gdf_localidades.crs)

print("\nArchivo Educación existe:", archivo_educacion.exists())
print("Archivo localidades existe:", archivo_localidades.exists())

=== CARGA PARA VALIDACIÓN ===
Educación: (747, 14)
Localidades: (20, 8)

CRS Educación: EPSG:3857
CRS Localidades: EPSG:4686

Archivo Educación existe: True
Archivo localidades existe: True


In [24]:
print("=== ESTRUCTURA DEL DATASET ===")

print("Filas:", gdf_educacion.shape[0])
print("Columnas:", gdf_educacion.shape[1])

print("\n=== COLUMNAS ===")
for i, columna in enumerate(gdf_educacion.columns, start=1):
    print(f"{i}. {columna}")

print("\n=== TIPOS DE DATOS ===")
print(gdf_educacion.dtypes)

=== ESTRUCTURA DEL DATASET ===
Filas: 747
Columnas: 14

=== COLUMNAS ===
1. NOMBRE_EST
2. GENERO
3. COD_LOCA
4. CLASE_TIPO
5. FECHA
6. OPreescola
7. OPrimaria
8. OSecundari
9. OMedia
10. OTotal
11. Aceleracio
12. DANE12_EST
13. Educacion_
14. geometry

=== TIPOS DE DATOS ===
NOMBRE_EST               str
GENERO                 int32
COD_LOCA                 str
CLASE_TIPO             int32
FECHA         datetime64[ms]
OPreescola             int32
OPrimaria              int32
OSecundari             int32
OMedia                 int32
OTotal                 int32
Aceleracio             int32
DANE12_EST               str
Educacion_             int32
geometry            geometry
dtype: object


In [25]:
print("=== VALORES NULOS POR COLUMNA ===")

nulos = pd.DataFrame({
    "nulos": gdf_educacion.isna().sum(),
    "porcentaje": (gdf_educacion.isna().mean() * 100).round(2)
})

display(nulos)

=== VALORES NULOS POR COLUMNA ===


,nulos,porcentaje
NOMBRE_EST,0,0.0
GENERO,0,0.0
COD_LOCA,0,0.0
CLASE_TIPO,0,0.0
FECHA,0,0.0
OPreescola,0,0.0
OPrimaria,0,0.0
OSecundari,0,0.0
OMedia,0,0.0
OTotal,0,0.0


In [26]:
print("=== DUPLICADOS EXACTOS ===")

duplicados_exactos = gdf_educacion.duplicated().sum()
porcentaje_duplicados = duplicados_exactos / len(gdf_educacion) * 100

print("Filas duplicadas exactas:", duplicados_exactos)
print(f"Porcentaje de duplicados: {porcentaje_duplicados:.4f}%")

=== DUPLICADOS EXACTOS ===
Filas duplicadas exactas: 0
Porcentaje de duplicados: 0.0000%


In [27]:
print("=== COHERENCIA DE OTOTAL ===")

componentes_oferta = [
    "OPreescola",
    "OPrimaria",
    "OSecundari",
    "OMedia",
    "Aceleracio",
    "Educacion_"
]

suma_componentes = gdf_educacion[componentes_oferta].sum(axis=1)

diferencia = gdf_educacion["OTotal"] - suma_componentes

print("Registros evaluados:", len(gdf_educacion))
print("Coinciden:", (diferencia == 0).sum())
print("No coinciden:", (diferencia != 0).sum())
print("Diferencia máxima absoluta:", diferencia.abs().max())

=== COHERENCIA DE OTOTAL ===
Registros evaluados: 747
Coinciden: 747
No coinciden: 0
Diferencia máxima absoluta: 0


In [28]:
print("\n=== VALORES NEGATIVOS ===")

columnas_oferta = componentes_oferta + ["OTotal"]

negativos = (gdf_educacion[columnas_oferta] < 0).sum()

print(negativos)
print("\nTotal de valores negativos:", negativos.sum())


=== VALORES NEGATIVOS ===
OPreescola    0
OPrimaria     0
OSecundari    0
OMedia        0
Aceleracio    0
Educacion_    0
OTotal        0
dtype: int64

Total de valores negativos: 0


In [29]:
print("=== CÓDIGOS DE LOCALIDAD ===")

codigos_localidad = sorted(
    gdf_educacion["COD_LOCA"].dropna().astype(str).unique()
)

print("Valores únicos:", codigos_localidad)
print("Cantidad de códigos distintos:", len(codigos_localidad))

codigos_esperados = {f"{i:02d}" for i in range(1, 21)}
codigos_observados = set(codigos_localidad)

print("\nCódigos esperados no encontrados:",
      sorted(codigos_esperados - codigos_observados))

print("Códigos fuera del rango esperado:",
      sorted(codigos_observados - codigos_esperados))

print("\nDistribución por localidad:")
print(
    gdf_educacion["COD_LOCA"]
    .value_counts()
    .sort_index()
)

=== CÓDIGOS DE LOCALIDAD ===
Valores únicos: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']
Cantidad de códigos distintos: 20

Códigos esperados no encontrados: []
Códigos fuera del rango esperado: []

Distribución por localidad:
COD_LOCA
01    27
02     7
03    16
04    63
05    73
06    23
07    63
08    75
09    21
10    69
11    69
12    23
13     3
14    14
15    12
16    32
17     3
18    50
19    81
20    23
Name: count, dtype: int64


In [30]:
print("=== CARDINALIDAD DE IDENTIFICADORES ===")

print("Registros:", len(gdf_educacion))
print(
    "Establecimientos por NOMBRE_EST:",
    gdf_educacion["NOMBRE_EST"].nunique()
)
print(
    "DANE12_EST únicos:",
    gdf_educacion["DANE12_EST"].nunique()
)
print(
    "Geometrías únicas:",
    gdf_educacion.geometry.nunique()
)

=== CARDINALIDAD DE IDENTIFICADORES ===
Registros: 747
Establecimientos por NOMBRE_EST: 408
DANE12_EST únicos: 412
Geometrías únicas: 741


In [31]:
print("=== CONSISTENCIA DANE - NOMBRE ===")

dane_nombre = (
    gdf_educacion
    .groupby("DANE12_EST")["NOMBRE_EST"]
    .nunique()
)

print(
    "DANE asociados a más de un nombre:",
    (dane_nombre > 1).sum()
)

display(
    dane_nombre[dane_nombre > 1]
    .sort_values(ascending=False)
)

=== CONSISTENCIA DANE - NOMBRE ===
DANE asociados a más de un nombre: 0


Series([], Name: NOMBRE_EST, dtype: int64)

In [32]:
print("=== CONSISTENCIA DANE - LOCALIDAD ===")

dane_localidad = (
    gdf_educacion
    .groupby("DANE12_EST")["COD_LOCA"]
    .nunique()
)

print(
    "DANE asociados a más de una localidad:",
    (dane_localidad > 1).sum()
)

display(
    dane_localidad[dane_localidad > 1]
    .sort_values(ascending=False)
)

=== CONSISTENCIA DANE - LOCALIDAD ===
DANE asociados a más de una localidad: 1


DANE12_EST
111001027332    2
Name: COD_LOCA, dtype: int64

In [33]:
print("=== DANE ASOCIADO A MÁS DE UNA LOCALIDAD ===")

dane_especial = "111001027332"

caso_dane = gdf_educacion.loc[
    gdf_educacion["DANE12_EST"].astype(str) == dane_especial,
    [
        "DANE12_EST",
        "NOMBRE_EST",
        "COD_LOCA",
        "GENERO",
        "CLASE_TIPO",
        "FECHA",
        "OTotal",
        "geometry"
    ]
].sort_values(["COD_LOCA", "OTotal"])

display(caso_dane)

=== DANE ASOCIADO A MÁS DE UNA LOCALIDAD ===


,DANE12_EST,NOMBRE_EST,COD_LOCA,GENERO,CLASE_TIPO,FECHA,OTotal,geometry
622,111001027332,COLEGIO GUSTAVO RESTREPO (IED),15,5,1,2025-03-31,372,POINT (-8248434.743 510811.471)
620,111001027332,COLEGIO GUSTAVO RESTREPO (IED),18,5,1,2025-03-31,783,POINT (-8249685.974 510274.308)
621,111001027332,COLEGIO GUSTAVO RESTREPO (IED),18,5,1,2025-03-31,1030,POINT (-8249061.472 510086.693)


In [34]:
print("=== DANE + GEOMETRÍA ===")

gdf_revision = gdf_educacion.copy()

gdf_revision["GEOM_KEY"] = gdf_revision.geometry.to_wkt()

combinaciones_dane_geometria = (
    gdf_revision[["DANE12_EST", "GEOM_KEY"]]
    .drop_duplicates()
    .shape[0]
)

duplicados_dane_geometria = gdf_revision.duplicated(
    subset=["DANE12_EST", "GEOM_KEY"]
).sum()

dane_geometrias = (
    gdf_revision
    .groupby("DANE12_EST")["GEOM_KEY"]
    .nunique()
)

print("Registros:", len(gdf_revision))
print(
    "Combinaciones únicas DANE + geometría:",
    combinaciones_dane_geometria
)
print(
    "Filas adicionales con DANE + geometría repetidos:",
    duplicados_dane_geometria
)
print(
    "DANE con múltiples geometrías:",
    (dane_geometrias > 1).sum()
)
print(
    "Máximo de geometrías para un mismo DANE:",
    dane_geometrias.max()
)

=== DANE + GEOMETRÍA ===
Registros: 747
Combinaciones únicas DANE + geometría: 743
Filas adicionales con DANE + geometría repetidos: 4
DANE con múltiples geometrías: 207
Máximo de geometrías para un mismo DANE: 14


In [35]:
print("=== REGISTROS CON DANE + GEOMETRÍA REPETIDOS ===")

repetidos_dane_geometria = gdf_revision.loc[
    gdf_revision.duplicated(
        subset=["DANE12_EST", "GEOM_KEY"],
        keep=False
    ),
    [
        "DANE12_EST",
        "NOMBRE_EST",
        "GENERO",
        "COD_LOCA",
        "CLASE_TIPO",
        "FECHA",
        "OPreescola",
        "OPrimaria",
        "OSecundari",
        "OMedia",
        "OTotal",
        "Aceleracio",
        "Educacion_",
        "geometry"
    ]
].sort_values(["DANE12_EST", "COD_LOCA"])

print("Filas involucradas:", len(repetidos_dane_geometria))

display(repetidos_dane_geometria)

=== REGISTROS CON DANE + GEOMETRÍA REPETIDOS ===
Filas involucradas: 8


,DANE12_EST,NOMBRE_EST,GENERO,COD_LOCA,CLASE_TIPO,FECHA,OPreescola,OPrimaria,OSecundari,OMedia,OTotal,Aceleracio,Educacion_,geometry
386,111001010740,COLEGIO INSTITUTO TECNICO INDUSTRIAL FRANCISCO...,5,10,1,2025-03-31,0,560,0,0,560,0,0,POINT (-8247655.507 521179.096)
387,111001010740,COLEGIO INSTITUTO TECNICO INDUSTRIAL FRANCISCO...,5,10,1,2025-03-31,301,140,0,0,441,0,0,POINT (-8247655.507 521179.096)
564,111001011274,COLEGIO ANDRES BELLO (IED),5,16,1,2025-03-31,0,317,560,320,1397,0,200,POINT (-8252148.362 512548.063)
565,111001011274,COLEGIO ANDRES BELLO (IED),5,16,1,2025-03-31,128,189,0,0,317,0,0,POINT (-8252148.362 512548.063)
200,111001018201,COLEGIO RUFINO JOSE CUERVO (IED),5,06,1,2025-03-31,36,240,0,0,276,0,0,POINT (-8252035.929 508047.521)
201,111001018201,COLEGIO RUFINO JOSE CUERVO (IED),5,06,1,2025-03-31,0,280,0,0,280,0,0,POINT (-8252035.929 508047.521)
221,111001046477,COLEGIO GRANCOLOMBIANO (IED),5,07,1,2025-03-31,257,0,1265,504,2026,0,0,POINT (-8259869.481 513763.141)
223,111001046477,COLEGIO GRANCOLOMBIANO (IED),5,07,1,2025-03-31,250,0,0,0,250,0,0,POINT (-8259869.481 513763.141)


In [36]:
print("=== VALIDACIÓN TERRITORIAL EDUCACIÓN ===")

# Trabajar en el mismo CRS de la capa oficial de localidades
educacion_territorial = gdf_educacion.to_crs(gdf_localidades.crs).copy()

localidades_ref = gdf_localidades[
    ["LocCodigo", "LocNombre", "geometry"]
].copy()

# Normalizar códigos territoriales a dos dígitos
educacion_territorial["COD_LOCA_NORM"] = (
    educacion_territorial["COD_LOCA"]
    .astype(str)
    .str.zfill(2)
)

localidades_ref["LocCodigo"] = (
    localidades_ref["LocCodigo"]
    .astype(str)
    .str.zfill(2)
)

resultado_territorial = gpd.sjoin(
    educacion_territorial,
    localidades_ref,
    how="left",
    predicate="within"
)

resultado_territorial["COINCIDE"] = (
    resultado_territorial["COD_LOCA_NORM"]
    == resultado_territorial["LocCodigo"]
)

sin_localidad = resultado_territorial["LocCodigo"].isna()
no_coincide = (
    ~resultado_territorial["COINCIDE"]
    & ~sin_localidad
)

print("Registros evaluados:", len(resultado_territorial))
print(
    "COD_LOCA coincide con geometría:",
    resultado_territorial["COINCIDE"].sum()
)
print(
    "COD_LOCA NO coincide con geometría:",
    no_coincide.sum()
)
print(
    "Sin localidad geométrica:",
    sin_localidad.sum()
)

=== VALIDACIÓN TERRITORIAL EDUCACIÓN ===
Registros evaluados: 747
COD_LOCA coincide con geometría: 743
COD_LOCA NO coincide con geometría: 3
Sin localidad geométrica: 1


In [37]:
print("=== CASOS TERRITORIALES A REVISAR ===")

casos_territoriales = resultado_territorial.loc[
    ~resultado_territorial["COINCIDE"],
    [
        "DANE12_EST",
        "NOMBRE_EST",
        "COD_LOCA",
        "LocCodigo",
        "LocNombre",
        "OTotal",
        "geometry"
    ]
].copy()

display(casos_territoriales)

=== CASOS TERRITORIALES A REVISAR ===


,DANE12_EST,NOMBRE_EST,COD_LOCA,LocCodigo,LocNombre,OTotal,geometry
182,211850001074,COLEGIO RURAL LAS MERCEDES (CED),05,19,CIUDAD BOLIVAR,44,POINT (-74.17622 4.39225)
244,111001107883,COLEGIO DEBORA ARANGO PEREZ (IED),07,08,KENNEDY,3009,POINT (-74.18348 4.61883)
590,111001013323,COLEGIO INTEGRADA LA CANDELARIA (IED),17,03,SANTA FE,329,POINT (-74.068 4.60151)
727,211001076346,COLEGIO CAMPESTRE JAIME GARZON (IED),20,NaN,NaN,31,POINT (-74.143 4.24)


In [38]:
print("=== VALIDEZ DE POLÍGONOS DE LOCALIDAD ===")

print(
    gdf_localidades.geometry
    .is_valid
    .value_counts(dropna=False)
)

=== VALIDEZ DE POLÍGONOS DE LOCALIDAD ===
True    20
Name: count, dtype: int64


In [39]:
print("=== DISTANCIAS DE CASOS TERRITORIALES ===")

# CRS métrico para medir distancias en metros
educacion_m = gdf_educacion.to_crs(epsg=3116).copy()
localidades_m = gdf_localidades.to_crs(epsg=3116).copy()

localidades_m["LocCodigo_NORM"] = (
    localidades_m["LocCodigo"]
    .astype(str)
    .str.zfill(2)
)

resultados_distancia = []

for idx in casos_territoriales.index.unique():

    fila = gdf_educacion.loc[idx]
    punto = educacion_m.loc[idx].geometry

    codigo_declarado = str(fila["COD_LOCA"]).zfill(2)

    # Distancia a todos los polígonos
    distancias = localidades_m.geometry.distance(punto)

    idx_cercana = distancias.idxmin()

    codigo_cercano = localidades_m.loc[
        idx_cercana, "LocCodigo_NORM"
    ]

    nombre_cercano = localidades_m.loc[
        idx_cercana, "LocNombre"
    ]

    distancia_cercana = distancias.loc[idx_cercana]

    # Localidad declarada por COD_LOCA
    localidad_declarada = localidades_m[
        localidades_m["LocCodigo_NORM"]
        == codigo_declarado
    ]

    if not localidad_declarada.empty:
        distancia_declarada = punto.distance(
            localidad_declarada.geometry.iloc[0]
        )
    else:
        distancia_declarada = None

    resultados_distancia.append({
        "DANE12_EST": fila["DANE12_EST"],
        "NOMBRE_EST": fila["NOMBRE_EST"],
        "COD_LOCA": codigo_declarado,
        "LOCALIDAD_MAS_CERCANA": codigo_cercano,
        "NOMBRE_LOCALIDAD_CERCANA": nombre_cercano,
        "DISTANCIA_CERCANA_M": round(
            distancia_cercana, 2
        ),
        "DISTANCIA_LOCALIDAD_DECLARADA_M": (
            None
            if distancia_declarada is None
            else round(distancia_declarada, 2)
        )
    })

distancias_territoriales = pd.DataFrame(
    resultados_distancia
)

display(distancias_territoriales)

=== DISTANCIAS DE CASOS TERRITORIALES ===


,DANE12_EST,NOMBRE_EST,COD_LOCA,LOCALIDAD_MAS_CERCANA,NOMBRE_LOCALIDAD_CERCANA,DISTANCIA_CERCANA_M,DISTANCIA_LOCALIDAD_DECLARADA_M
0,211850001074,COLEGIO RURAL LAS MERCEDES (CED),05,19,CIUDAD BOLIVAR,0.00,401.49
1,111001107883,COLEGIO DEBORA ARANGO PEREZ (IED),07,08,KENNEDY,0.00,30.88
2,111001013323,COLEGIO INTEGRADA LA CANDELARIA (IED),17,03,SANTA FE,0.00,0.69
3,211001076346,COLEGIO CAMPESTRE JAIME GARZON (IED),20,20,SUMAPAZ,219.33,219.33


## Notas de validación — Educación

- El dataset contiene **747 registros y 14 variables**.
- El archivo se encuentra estructurado como un `GeoDataFrame` con sistema de referencia espacial **EPSG:3857**.
- Se identificaron **747 geometrías tipo `Point`**, sin geometrías nulas ni vacías.
- No se encontraron valores nulos en ninguna de las variables del dataset.
- No se encontraron duplicados exactos.
- Las variables cuantitativas asociadas a la oferta de cupos no presentan valores negativos.

### Coherencia de la oferta

- La variable `OTotal` fue contrastada con la suma de:
  - `OPreescola`
  - `OPrimaria`
  - `OSecundari`
  - `OMedia`
  - `Aceleracio`
  - `Educacion_`
- Los **747 registros** presentan correspondencia exacta entre `OTotal` y la suma de sus componentes.
- No se identificaron diferencias en esta validación.

### Cobertura territorial declarada

- La variable `COD_LOCA` contiene los **20 códigos de localidad esperados (`01` a `20`)**.
- No se identificaron códigos faltantes ni valores por fuera del rango esperado.
- La cobertura territorial declarada incluye, por tanto, las 20 localidades de Bogotá.

### Identificadores de establecimientos

- Registros totales: **747**.
- Nombres de establecimiento distintos (`NOMBRE_EST`): **408**.
- Identificadores `DANE12_EST` distintos: **412**.
- Geometrías distintas: **741**.
- Ningún `DANE12_EST` se encuentra asociado a más de un nombre de establecimiento.
- Se identificó **un `DANE12_EST` asociado a más de una localidad**: `111001027332`, correspondiente al **COLEGIO GUSTAVO RESTREPO (IED)**.

La presencia de múltiples registros para un mismo establecimiento o código DANE no se interpreta automáticamente como duplicación. El dataset presenta una granularidad superior a una fila por establecimiento, y existen registros con diferencias en variables de oferta y/o ubicación.

### Consistencia DANE y geometría

- Se identificaron **743 combinaciones únicas de `DANE12_EST` + geometría**.
- Existen registros que comparten identificador DANE y posición geográfica, pero difieren en otras variables del dataset.
- Se identificaron establecimientos con múltiples geometrías asociadas, por lo que la geometría tampoco debe utilizarse de manera aislada como clave única.
- Estas observaciones no justifican la eliminación automática de registros.

### Validación territorial

Para contrastar `COD_LOCA` con la ubicación espacial de cada registro se utilizó como referencia la capa oficial de localidades almacenada en `data/external/loca.json`.

La capa de localidades contiene **20 polígonos válidos**.

Resultados del contraste espacial:

- Registros evaluados: **747**.
- `COD_LOCA` coincidente con la localidad determinada espacialmente: **743**.
- Registros cuya geometría intersecta una localidad diferente a la declarada en `COD_LOCA`: **3**.
- Registros sin asignación directa mediante la operación espacial `within`: **1**.

### Casos territoriales a revisar

Se identificaron cuatro casos especiales:

1. **COLEGIO RURAL LAS MERCEDES (CED)**
   - `COD_LOCA` declarado: **05 — Usme**.
   - La geometría se localiza espacialmente en **19 — Ciudad Bolívar**.
   - Distancia aproximada al polígono correspondiente a la localidad declarada: **401,49 m**.
   - Se registra como una discrepancia territorial que requiere revisión de la fuente.

2. **COLEGIO DEBORA ARANGO PEREZ (IED)**
   - `COD_LOCA` declarado: **07 — Bosa**.
   - La geometría se localiza espacialmente en **08 — Kennedy**.
   - Distancia aproximada al polígono correspondiente a la localidad declarada: **30,88 m**.
   - Debido a su proximidad al límite territorial, se conserva como caso de revisión y no se corrige automáticamente.

3. **COLEGIO INTEGRADA LA CANDELARIA (IED)**
   - `COD_LOCA` declarado: **17 — La Candelaria**.
   - La geometría se localiza espacialmente en **03 — Santa Fe**.
   - Distancia aproximada al polígono correspondiente a la localidad declarada: **0,69 m**.
   - La distancia extremadamente pequeña indica un caso limítrofe, por lo que no se considera evidencia suficiente para modificar el registro original.

4. **COLEGIO CAMPESTRE JAIME GARZON (IED)**
   - `COD_LOCA` declarado: **20 — Sumapaz**.
   - El punto no queda contenido directamente dentro de ninguno de los polígonos mediante `within`.
   - La localidad más cercana es **20 — Sumapaz**, a aproximadamente **219,33 m**.
   - Se mantiene el código territorial original y el caso se documenta como una anomalía espacial que requiere revisión posterior.

### Conclusión de validación

El dataset presenta una calidad estructural adecuada para continuar dentro del flujo de trabajo:

- No presenta valores nulos.
- No presenta duplicados exactos.
- La oferta total es matemáticamente coherente en los 747 registros.
- No existen valores negativos en las variables de oferta.
- Los 20 códigos de localidad esperados están presentes.
- La mayoría de los registros (**743 de 747**) presenta coherencia directa entre `COD_LOCA` y la geometría.

Las excepciones territoriales detectadas se conservan y documentan sin modificar el archivo original.

> La validación territorial realizada en este notebook constituye un control de calidad de los atributos existentes. No representa todavía una integración territorial del dominio Educación con otros dominios del proyecto.

> No se realizan correcciones automáticas sobre `COD_LOCA`, geometrías, identificadores DANE ni variables de oferta. Cualquier normalización o integración posterior deberá realizarse en una etapa distinta y mantener trazabilidad respecto al dato crudo.